In [1]:
import numpy as np
import pandas as pd
import csv
import matplotlib.pyplot as plt
import xlrd

%matplotlib inline
%matplotlib notebook

### 1. Load csv file

In [2]:
path = 'C:/Users/ZWX/PythonNotebooks/UWBM/Unittest/PavedRoof/'
InputData = pd.read_csv(path + 'input_csv.csv')

In [3]:
date = InputData['date']
P_atm = InputData['P_atm']
Ref_grass = InputData['Ref.grass']
E_pot_OW = InputData['E_pot_OW']

### 2. Paved roof ###

In [4]:
# iters = Total timestep.
iters = np.shape(date)[0] 

#### 2.1 General test (Default settings)

In [5]:
# Given conditions.

# tot_area --- total area of study domain
tot_area = 10000

# pr_frac --- pavedroof percentage
pr_frac = 0.156  
tot_pr_area = tot_area * pr_frac

# pr_meas_area --- area of measure on pavedroof
# pr_nomeas_area --- area of pavedroof (without a measure)
pr_meas_area = 0  
pr_no_meas_area = tot_pr_area - pr_meas_area


# inflowfac --- inflow factor
# measure_inflow_area --- runoff inflow area to measure, inflow area >= measure area, predefined as 0.
measure_inflow_area = 0 
inflowfac_pr = (measure_inflow_area - pr_meas_area) / pr_no_meas_area

In [6]:
class PavedRoof:
    def __init__(self, init_intstor_pr, intstorcap_pr = 1.6, stormfrac_pr = 1.0, discfrac_pr = 0.0):
        
        # state
        # init_intstor_pr --- initial interception storage
        
        self.init_intstor_pr = init_intstor_pr
        
        # parameters
        # intstorcap_pr --- predefined storage capacity on pavedroof
        # stormfrac_pr --- part of urban area with storm water drainage system
        # discfrac_pr --- part of paved roof area that is disconnected
        # self.mxdfrac--- part of urban area with mixed sewer system
        
        self.intstorcap = intstorcap_pr
        self.stormfrac = stormfrac_pr
        self.mxdfrac = 1 - self.stormfrac
        self.discfrac = discfrac_pr
        
    def __repr__(self):
        return 'Current P is ' + str(p_atm) + 'Current E is ' + str(e_pot_ow) + '.These are input information.'
               
    def sol(self, p_atm, e_pot_ow):
        
        # int_pr --- Interception on paved roof after rainfall during current time step [mm]
        # e_atm_pr --- Evaporation from interception storage on paved roof during current time step [mm]
        # intstor_pr --- Remaining interception storage on paved roof at the end of the current time step [mm]
        # r_pr_meas --- Runoff from paved roof to an area with a drainage measure (not necessarily on the roof itself) [mm].
        # r_pr_swds --- Runoff from paved roof to the storm water drainage system [mm]
        # r_pr_mss --- Runoff from paved roof to the mixed sewer system [mm]
        # r_pr_up --- Runoff from paved roof to unpaved area [mm].
        
        
        if pr_no_meas_area == 0:
            int_pr = e_atm_pr = intstor_pr = r_pr_meas = r_pr_swds = r_pr_mss = r_pr_up = 0
            
        else:
            int_pr = min(self.intstorcap, max(0, self.init_intstor_pr + p_atm))
            
            e_atm_pr = min(e_pot_ow, int_pr)
            
            intstor_pr = int_pr - e_atm_pr
            
            r_pr_meas = inflowfac_pr * max(0, p_atm - e_atm_pr - (intstor_pr - self.init_intstor_pr))
            
            r_pr_swds = self.stormfrac * (1 - self.discfrac) * max(0, p_atm - e_atm_pr - (intstor_pr - self.init_intstor_pr) - r_pr_meas)
            
            r_pr_mss = self.mxdfrac * (1 - self.discfrac) * max(0, p_atm - e_atm_pr - (intstor_pr - self.init_intstor_pr) - r_pr_meas)
            
            r_pr_up = self.discfrac * max(0, p_atm - e_atm_pr - (intstor_pr - self.init_intstor_pr) - r_pr_meas)
            
            # update state
            self.init_intstor_pr = intstor_pr
        
        return int_pr, e_atm_pr, intstor_pr, r_pr_meas, r_pr_swds, r_pr_mss, r_pr_up

In [7]:
# Database for all unknown. ([0] is used to fill the vacancy at time level t = 0.) 

E_atm = [0]
Intcp = [0]
IntStor = [0]
R_meas = [0]
R_swds = [0]
R_mss = [0]
R_up = [0]

m = PavedRoof(init_intstor_pr = 0, intstorcap_pr = 1.6, stormfrac_pr = 1.0, discfrac_pr = 0.0)

t = 1

while t <= iters-1:
    # only loop sol(), not repeat creating new object.
    sol = m.sol(P_atm[t], E_pot_OW[t])
    
    Intcp.append(sol[0])
    E_atm.append(sol[1])
    IntStor.append(sol[2])
    R_meas.append(sol[3])
    R_swds.append(sol[4])
    R_mss.append(sol[5])
    R_up.append(sol[6])
    
    # print('time step', t)
    t += 1
    
filename = 'General_test_pysol.csv'
np.savetxt('pysol/' + filename, np.c_[Intcp, E_atm, IntStor, R_meas, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Intcp, E_atm, IntStor, R_meas, R_swds, R_mss, R_up')#np.c_ is used here to convert horizontal into vertical structure
# Insert the Date column for locating purposes.
df = pd.read_csv('pysol/' + filename)
#date_column = pd.DataFrame({'Date': date})
#df = df.merge(date_column, left_index = True, right_index = True)
df.insert(0, 'Date', date)
df.to_csv('pysol/' + filename)

##### 2.1.1 Validation

___Methodology:___ 

Take the difference between python sol and excel sol at each timestep for every term, to see whether max(diff) = min(diff) = 0. If the errors are 10 to the power of -10. Then the results are validated.

___Note___
I came across some chanllenges on dealing with the data.type in excel, and it took me quite a long time to think. But still I didn't manage to solve it. so in order to avoid that I choose another way of reading excel sol, which is much more efficient and simpler, and that it is store the particular part of results in excel in a csv file.

In [8]:
# read python file
data_py = pd.read_csv('pysol/' + filename)

In [9]:
# read excel file
data_ex = pd.read_csv('exsol/General_test_exsol.csv')

In [10]:
# Examine (go through all the data)
database = []
# from col 0 to last col (col 6)
for c in range(7):
    for r in range(1,43825): # from row 1 to the last row (row 43824)
        a = data_ex[list(data_ex)[c]][r] - data_py[list(data_py)[c+2]][r]
        database.append(a)
print(max(database))
print(min(database))

1.000000082740371e-08
-5.000000413701855e-09


In [11]:
# Examine (go through column by colum)
A = np.zeros((43825, 7))
for c in range(7):
    for r in range(1,43825): # do not include the initial row.
        A[r,c] = data_ex[list(data_ex)[c]][r] - data_py[list(data_py)[c+2]][r]
for c in range(7):
    print('col ' + str(c), 'max', max(A[:,c]), 'min', min(A[:,c]))

col 0 max 5.00000019165725e-09 min -5.00000019165725e-09
col 1 max 5.000000025123796e-09 min -5.000000025123796e-09
col 2 max 5.00000019165725e-09 min -5.00000019165725e-09
col 3 max 0.0 min 0.0
col 4 max 1.000000082740371e-08 min -5.000000413701855e-09
col 5 max 0.0 min 0.0
col 6 max 0.0 min 0.0


#### 2.2 Extended test (Different coefficient sets)

Set 1: pr_no_meas_area = 0

Set 2: intstorcap_pr = 0

Set 3: intstorcap_pr = 600

Set 4: stormfrac = 0.0

Set 5: stormfrac = 0.37

Set 6: discfrac = 1.0

Set 7: discfrac = 0.37

__Note:__ Build a validatefunc() to easily manipulate results without introducing too many cells.

a---intstorcap_pr

b---stormfrac_pr

c---discfrac_pr

d---set number

In [12]:
def validatefunc(a, b, c, d): # d is the set number
    
    E_atm = [0]
    Intcp = [0]
    IntStor = [0]
    R_meas = [0]
    R_swds = [0]
    R_mss = [0]
    R_up = [0]

    m = PavedRoof(init_intstor_pr = 0, intstorcap_pr = a, stormfrac_pr = b, discfrac_pr = c)

    t = 1

    while t <= iters-1:
        # only loop sol(), not repeat creating new object.
        sol = m.sol(P_atm[t], E_pot_OW[t])
    
        Intcp.append(sol[0])
        E_atm.append(sol[1])
        IntStor.append(sol[2])
        R_meas.append(sol[3])
        R_swds.append(sol[4])
        R_mss.append(sol[5])
        R_up.append(sol[6])
    
        # print('time step', t)
        t += 1
    
    filename = 'PR_extended_test_pysol_set'+str(d)+'.csv'
    np.savetxt('pysol/' + filename, np.c_[Intcp, E_atm, IntStor, R_meas, R_swds, R_mss, R_up], fmt = "%.8f", delimiter=',', header = 'Intcp, E_atm, IntStor, R_meas, R_swds, R_mss, R_up')
    df = pd.read_csv('pysol/' + filename)
    df.insert(0, 'Date', date)
    df.to_csv('pysol/' + filename)
    data_py = pd.read_csv('pysol/' + filename)
    data_ex = pd.read_csv('exsol/PR_extended_test_exsol_set'+str(d)+'.csv')
    A = np.zeros((43825, 7))
    for c in range(7):
        for r in range(1,43825):
            A[r,c] = data_ex[list(data_ex)[c]][r] - data_py[list(data_py)[c+2]][r]
    for c in range(7):
        print('col ' + str(c), 'max', max(A[:,c]), 'min', min(A[:,c]))
        #print(np.where(max(A[:,c]) != 0 and A[:,c] == max(A[:,c])), np.where(min(A[:,c]) != 0 and A[:,c] == min(A[:,c])))
    return 

__a---intstorcap_pr, b---stormfrac_pr ,c---discfrac_pr, d---set number__

##### 2.2.1 Set 1:  pr_no_meas_area = 0

In [13]:
pr_no_meas_area = 0
validatefunc(1.6, 1.0, 0.0, 1)

col 0 max 0.0 min 0.0
col 1 max 0.0 min 0.0
col 2 max 0.0 min 0.0
col 3 max 0.0 min 0.0
col 4 max 0.0 min 0.0
col 5 max 0.0 min 0.0
col 6 max 0.0 min 0.0


##### 2.2.2 Set 2: intstorcap_pr = 0

In [14]:
pr_no_meas_area = 1560
validatefunc(0, 1, 0, 2)

col 0 max 0.0 min 0.0
col 1 max 0.0 min -3.0007100000000002e-15
col 2 max 3.0007100000000002e-15 min 0.0
col 3 max 0.0 min 0.0
col 4 max 3.0007100000000002e-15 min 0.0
col 5 max 0.0 min 0.0
col 6 max 0.0 min 0.0


##### 2.2.3 Set 3: intstorcap_pr = 1600

In [15]:
validatefunc(1600, 1, 0, 3)

col 0 max 5.000001692678779e-07 min -5.000001692678779e-07
col 1 max 5.000000025123796e-09 min -5.000000025123796e-09
col 2 max 5.000001692678779e-07 min -5.000001692678779e-07
col 3 max 0.0 min 0.0
col 4 max 5.000000025123796e-09 min -5.000000025123796e-09
col 5 max 0.0 min 0.0
col 6 max 0.0 min 0.0


##### 2.2.4 Set 4: stormfrac = 0.0

In [16]:
validatefunc(1.6, 0, 0, 4)

col 0 max 5.00000019165725e-09 min -5.00000019165725e-09
col 1 max 5.000000025123796e-09 min -5.000000025123796e-09
col 2 max 5.00000019165725e-09 min -5.00000019165725e-09
col 3 max 0.0 min 0.0
col 4 max 0.0 min 0.0
col 5 max 1.000000082740371e-08 min -5.000000413701855e-09
col 6 max 0.0 min 0.0


##### 2.2.5 Set 5: stormfrac = 0.37

In [17]:
validatefunc(1.6, 0.37, 0, 5)

col 0 max 5.00000019165725e-09 min -5.00000019165725e-09
col 1 max 5.000000025123796e-09 min -5.000000025123796e-09
col 2 max 5.00000019165725e-09 min -5.00000019165725e-09
col 3 max 0.0 min 0.0
col 4 max 5.000000413701855e-09 min -5.000000413701855e-09
col 5 max 5.000000413701855e-09 min -5.000000413701855e-09
col 6 max 0.0 min 0.0


##### 2.2.6 Set 6: discfrac = 1.0

In [18]:
validatefunc(1.6, 1, 1, 6)

col 0 max 5.00000019165725e-09 min -5.00000019165725e-09
col 1 max 5.000000025123796e-09 min -5.000000025123796e-09
col 2 max 5.00000019165725e-09 min -5.00000019165725e-09
col 3 max 0.0 min 0.0
col 4 max 0.0 min 0.0
col 5 max 0.0 min 0.0
col 6 max 1.000000082740371e-08 min -5.000000413701855e-09


##### 2.2.7 Set 7: discfrac = 0.37

In [19]:
validatefunc(1.6, 1, 0.37, 7)

col 0 max 5.00000019165725e-09 min -5.00000019165725e-09
col 1 max 5.000000025123796e-09 min -5.000000025123796e-09
col 2 max 5.00000019165725e-09 min -5.00000019165725e-09
col 3 max 0.0 min 0.0
col 4 max 5.000000413701855e-09 min -5.000000413701855e-09
col 5 max 0.0 min 0.0
col 6 max 5.000000413701855e-09 min -5.000000413701855e-09


In [20]:
print('The module has been validated in both general test and extended tests')

The module has been validated in both general test and extended tests
